# DeepFilterNet df_mlx `train_dynamic` (Google Colab)

This notebook sets up a Colab runtime, installs DeepFilterNet, and runs `python -m df_mlx.train_dynamic`.

Before running:
- In Colab, switch runtime to **GPU** (`Runtime -> Change runtime type`).
- Put your dataset config JSON (or cache dir) on Google Drive.
- Update the path variables in Cell 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Update these paths for your environment
REPO_URL = 'https://github.com/sealad886/DeepFilterNet4.git'
REPO_DIR = Path('/content/DeepFilterNet')
PROJECT_DIR = REPO_DIR / 'DeepFilterNet'
VENV_DIR = REPO_DIR / '.venv'
VENV_PY = VENV_DIR / 'bin' / 'python'

# Data source: use either DATASET_CONFIG_JSON or CACHE_DIR
DATASET_CONFIG_JSON = Path('/content/drive/MyDrive/DeepFilterNet/data/config.json')
CACHE_DIR = None  # Example: Path('/content/drive/MyDrive/DeepFilterNet/cache/mlx_datastore')

# Optional run-config TOML
RUN_CONFIG_TOML = None  # Example: Path('/content/drive/MyDrive/DeepFilterNet/configs/run_config.toml')

# Checkpoints and run shape
CHECKPOINT_DIR = Path('/content/drive/MyDrive/DeepFilterNet/checkpoints/df_mlx_train_dynamic')
EPOCHS = 5
BATCH_SIZE = 8
BACKBONE = 'mamba'  # mamba | gru | attention
PRESET = 'pro'      # entry | pro | max | ultra | debug

# Extra CLI flags (space-separated string), examples:
# '--dynamic-loss pipeline_awesome --resume --resume-data --checkpoint-batches 500'
EXTRA_FLAGS = '--resume --resume-data --checkpoint-batches 500'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('Configuration ready.')
print('REPO_DIR:', REPO_DIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)

In [ ]:
%%bash
set -euo pipefail

if [[ ! -d /content/DeepFilterNet/.git ]]; then
  git clone --depth 1 https://github.com/sealad886/DeepFilterNet4.git /content/DeepFilterNet
else
  git -C /content/DeepFilterNet fetch --all --prune
  git -C /content/DeepFilterNet pull --ff-only
fi

apt-get update -y
apt-get install -y python3.10 python3.10-venv libsndfile1 ffmpeg cargo libportaudio2 python3-pyaudio libportaudio-ocaml-dev

In [ ]:
%%bash
set -euo pipefail

# Define variables from Python cell x2mXbtMLggGT
REPO_DIR="/content/DeepFilterNet"
VENV_DIR="${REPO_DIR}/.venv"
VENV_PY="${VENV_DIR}/bin/python"

echo "REPO_DIR: ${REPO_DIR}"
echo "VENV_DIR: ${VENV_DIR}"
echo "VENV_PY: ${VENV_PY}"

# Create virtual environment
echo "$ python3.10 -m venv ${VENV_DIR}"
python3.10 -m venv "${VENV_DIR}"

# Upgrade pip, setuptools, wheel
echo "$ ${VENV_PY} -m pip install --upgrade pip setuptools wheel"
"${VENV_PY}" -m pip install --upgrade pip setuptools wheel

# Run setup.sh, changing directory first
echo "$ bash setup.sh --all --venv ${VENV_DIR}"
(cd "${REPO_DIR}" && bash setup.sh --all --venv "${VENV_DIR}")

# Determine MLX package based on GPU presence
MLX_PKG=""
if command -v nvidia-smi &> /dev/null; then
    MLX_PKG="mlx[cuda12]"
    echo "GPU detected, installing ${MLX_PKG}"
else
    MLX_PKG="mlx[cpu]"
    echo "No GPU detected, installing ${MLX_PKG}"
fi

# Install MLX explicitly for Colab
echo "$ ${VENV_PY} -m pip install --upgrade ${MLX_PKG}"
"${VENV_PY}" -m pip install --upgrade "${MLX_PKG}"

# Verify imports used by train_dynamic
echo "$ ${VENV_PY} -c 'import mlx, df_mlx.train_dynamic; print(\"mlx+df_mlx import OK\")'"
"${VENV_PY}" -c 'import mlx, df_mlx.train_dynamic; print("mlx+df_mlx import OK")'

In [ ]:
import os
import shlex
import subprocess

if CACHE_DIR is None and not DATASET_CONFIG_JSON.exists():
    raise FileNotFoundError(f'Dataset config not found: {DATASET_CONFIG_JSON}')

cmd = [
    str(VENV_PY),
    '-m',
    'df_mlx.train_dynamic',
    '--checkpoint-dir',
    str(CHECKPOINT_DIR),
    '--epochs',
    str(EPOCHS),
    '--batch-size',
    str(BATCH_SIZE),
    '--backbone-type',
    BACKBONE,
    '--preset',
    PRESET,
]

if CACHE_DIR is not None:
    cmd.extend(['--cache-dir', str(CACHE_DIR)])
else:
    cmd.extend(['--config', str(DATASET_CONFIG_JSON)])

if RUN_CONFIG_TOML is not None:
    run_cfg = Path(RUN_CONFIG_TOML)
    if not run_cfg.exists():
        raise FileNotFoundError(f'Run config not found: {run_cfg}')
    cmd.extend(['--run-config', str(run_cfg)])

if EXTRA_FLAGS.strip():
    cmd.extend(shlex.split(EXTRA_FLAGS))

print('Training command:')
print(' '.join(shlex.quote(part) for part in cmd))

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['DFNET_TQDM'] = '1'

print('Starting training...')
subprocess.run(cmd, cwd=str(PROJECT_DIR), env=env, check=True)

ckpt_path = Path(CHECKPOINT_DIR)
if not ckpt_path.exists():
    print('No checkpoint directory found yet.')
else:
    print('Recent checkpoint artifacts:')
    files = sorted(ckpt_path.glob('*'), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in files[:20]:
        print(p.name)